In [103]:
import torch 


path_celeba = "/home/nicola.debole/projects/Argus/argus/annotated_datasets/TRAIN-celeba-898-entropy-sum-42_2025_10_20_13_52.pt"
path_s3d = "/home/nicola.debole/projects/Argus/argus/annotated_datasets/TRAIN-shapes3d-840-random-sum-45_2025_10_09_03_29.pt"

dataset_train = torch.load(path_celeba)
print(len(dataset_train))
print(dataset_train[0])

25000
(None, tensor([0.1018, 0.1278, 0.4486, 0.1755, 0.0040, 0.4094, 0.1188, 0.3639, 0.6158,
        0.0070, 0.0171, 0.0744, 0.3372, 0.0333, 0.0050, 0.0334, 0.0886, 0.0026,
        0.0109, 0.1330, 0.2380, 0.1121, 0.1043, 0.8621, 0.2384, 0.1547, 0.0763,
        0.0186, 0.0021, 0.0539, 0.0986, 0.4055, 0.2060, 0.0240, 0.0287, 0.0272,
        0.0348, 0.0141, 0.9737], device='cuda:0'), tensor([0.3023, 0.3339, 0.4974, 0.3804, 0.0632, 0.4917, 0.3235, 0.4811, 0.4864,
        0.0834, 0.1295, 0.2625, 0.4728, 0.1793, 0.0706, 0.1797, 0.2841, 0.0509,
        0.1038, 0.3395, 0.4259, 0.3155, 0.3056, 0.3448, 0.4261, 0.3617, 0.2654,
        0.1352, 0.0458, 0.2258, 0.2982, 0.4910, 0.4044, 0.1530, 0.1669, 0.1627,
        0.1834, 0.1181, 0.1601], device='cuda:0'), tensor(1))


In [104]:
from CQA.datasets import GenericDataset

celeba_ds = GenericDataset(ds_name="celeba", split='train')
shapes3d_ds = GenericDataset(ds_name="shapes3d", split='train')

labels = []
g = []
for sample in celeba_ds:
    _,c,l = sample
    labels.append(l)
    g.append(c)
    
labels = torch.stack(labels, dim=0)
g = torch.stack(g,dim=0)

2025-10-30 15:47:16.207 | DEBUG    | CQA.datasets:get_dataset:31 - Getting dataset celeba with kwargs {'split': 'train'}
2025-10-30 15:47:26.908 | DEBUG    | CQA.datasets:__init__:68 - /mnt/cimec-storage6/shared/cv_datasets/celeba_manual_download/dataset_info.json
2025-10-30 15:47:26.909 | DEBUG    | CQA.datasets:__init__:70 - Loading dataset celeba from /mnt/cimec-storage6/shared/cv_datasets/celeba_manual_download
2025-10-30 15:47:26.911 | DEBUG    | CQA.datasets:__init__:72 - Loading frequencies
2025-10-30 15:47:26.920 | DEBUG    | CQA.datasets:get_dataset:31 - Getting dataset shapes3d with kwargs {'split': 'train'}
2025-10-30 15:47:28.041 | DEBUG    | CQA.datasets:__init__:68 - /mnt/cimec-storage6/shared/cv_datasets/shapes3d/dataset_info.json
2025-10-30 15:47:28.042 | DEBUG    | CQA.datasets:__init__:70 - Loading dataset shapes3d from /mnt/cimec-storage6/shared/cv_datasets/shapes3d
2025-10-30 15:47:28.043 | DEBUG    | CQA.datasets:__init__:72 - Loading frequencies


In [105]:
# label frequencies
p = torch.sum(labels)/len(labels)
p_y0 = 1-p.item()
p_y1 = p.item()
print(p_y0,p_y1)

0.584960013628006 0.415039986371994


In [106]:
sample_id=0
sample_label = labels[sample_id].item()
C_map = (dataset_train[sample_id][1] > 0.5).to(torch.int).cpu()
gt = g[sample_id]
print(C_map)
print(gt)
print(labels[sample_id])

tensor([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], dtype=torch.int32)
tensor([0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1])
tensor(1)


In [107]:
# concept conditional frequencies
num_samples = len(g)
concepts_given_0 = []
concepts_given_1 = []
for i in range(num_samples):
    if labels[i] == 0:
        concepts_given_0.append(g[i])
    else:
        concepts_given_1.append(g[i])
concepts_given_0 = torch.stack(concepts_given_0, dim=0)
concepts_given_1 = torch.stack(concepts_given_1, dim=0)
print((torch.sum(concepts_given_0,dim=0)/num_samples)[0:10])
print((torch.sum(concepts_given_1,dim=0)/num_samples)[0:10])

tensor([4.0000e-05, 2.4112e-01, 4.0096e-01, 5.8440e-02, 4.0000e-05, 1.1724e-01,
        1.7424e-01, 6.0160e-02, 1.1464e-01, 1.3832e-01])
tensor([0.1099, 0.0227, 0.1151, 0.1428, 0.0216, 0.0379, 0.0656, 0.1748, 0.1228,
        0.0082])


In [123]:

def c_given_y(c_state, y_state):
    if y_state==0:
        p_c1 = torch.sum(concepts_given_0,dim=0)/num_samples
    else:
        p_c1 = torch.sum(concepts_given_1,dim=0)/num_samples
    #print((torch.sum(concepts_given_0,dim=0)/num_samples)[0:10])
    if c_state == 1:
        return p_c1
    else:
        return 1-p_c1

def ratio(C_map, index, y_true):
    r = p_y1/p_y0
    # compute p(C_i) using naive bayes
    naiv_bayes_tar = c_given_y(1-C_map[index], 0)[index] + r*c_given_y(1-C_map[index], 1)[index]
    print(C_map[index])
    print(c_given_y(C_map[index], 0)[index])
    print(c_given_y(C_map[index], 1)[index])
    naiv_bayes_map = c_given_y(C_map[index], 0)[index] + r*c_given_y(C_map[index], 1)[index]
    print(f"P(c={C_map[index]}|y=0){c_given_y(C_map[index], 0)[index]}")
    print(f"P(c={C_map[index]}|y=1){c_given_y(C_map[index], 1)[index]}")
    print(f"P(c={1-C_map[index]}|y=0){c_given_y(1-C_map[index], 0)[index]}")
    print(f"P(c={1-C_map[index]}|y=1){c_given_y(1-C_map[index], 1)[index]}")
    print(f"P(...,c={C_map[index]},..)/P(...,c={1-C_map[index]},...) = {naiv_bayes_map/naiv_bayes_tar}")
    prior = naiv_bayes_map/naiv_bayes_tar
    
    if prior > 1:
        print(("The MAP world is more likely then TAR"))
    else:
        print(("The TAR world is more likely then MAP"))
    c_state = C_map[index]
    print(f"P(c={c_state})={naiv_bayes_map}, P(c={1-c_state})={naiv_bayes_tar})")
    if c_state == 1:
        print(f"P(c=1|x)={dataset_train[sample_id][1][index].cpu()}")
        print(f"P(c=1|y={y_true})={c_given_y(c_state, y_true)[index]}")
        num_map = c_given_y(c_state, y_true)[index]*dataset_train[sample_id][1][index].cpu()
        num_tar = c_given_y(1-C_map[index], y_true)[index]*(1-dataset_train[sample_id][1][index].cpu())
        print(f"map:{c_given_y(c_state, y_true)[index]:.2f}/{naiv_bayes_map}*{dataset_train[sample_id][1][index].cpu():.2f}")
    else:
        print(f"P(c=0|x)={1-dataset_train[sample_id][1][index].cpu()}")
        print(f"P(c=0|y={y_true})={c_given_y(c_state, y_true)[index]}")
        print(f"P(c=1|y={y_true})={c_given_y(1, y_true)[index]}")
        num_map = c_given_y(c_state, y_true)[index]*(1-dataset_train[sample_id][1][index].cpu())
        num_tar = c_given_y(1-C_map[index], y_true)[index]*dataset_train[sample_id][1][index].cpu()
        print(f"map:{c_given_y(c_state, y_true)[index]:.2f}/{naiv_bayes_map}*{1-dataset_train[sample_id][1][index].cpu():.2f}")
   
    
    
    map = num_map/naiv_bayes_map
    tar = num_tar/naiv_bayes_tar
    
    posterior = num_map/num_tar
    if posterior > 1:
        print(("The MAP posterior is more likely then TAR"))
    else:
        print(("The TAR posterior is more likely then MAP"))
    res = map/tar
    print(f"P(c|x,y) = {map:.2f}/{tar.item():.2f} = {res:.3f}")
    
    #print(res)
    return res.item()

def interpret_ratio(ratio, pred):
    if ratio<1:
        return f"Reject {pred} and use {1-pred}" 
    else:
        return f"Keep {pred}."

In [124]:
id_to_change = []
for c_id in range(39):
    #if not (c_id < 10 or c_id == 36):
    #    continue
    print(f"\n------------[{c_id}]--------------")
    r = ratio(C_map, c_id, sample_label)
    #print(r)
    #print(c_id,interpret_ratio(ratio(C_map, c_id, sample_label), C_map[c_id]))
    if r <= 1:
        id_to_change.append(c_id)
    
print(id_to_change)
print(C_map)
print(gt.to(torch.int), sample_label)
    


------------[0]--------------
tensor(0, dtype=torch.int32)
tensor(1.0000)
tensor(0.8901)
P(c=0|y=0)0.9999600052833557
P(c=0|y=1)0.8901200294494629
P(c=1|y=0)3.9999998989515007e-05
P(c=1|y=1)0.1098800003528595
P(...,c=0,..)/P(...,c=1,...) = 20.916370391845703
The MAP world is more likely then TAR
P(c=0)=1.631516695022583, P(c=1)=0.07800190150737762)
P(c=0|x)=0.8982337713241577
P(c=0|y=1)=0.8901200294494629
P(c=1|y=1)=0.1098800003528595
map:0.89/1.631516695022583*0.90
The MAP posterior is more likely then TAR
P(c|x,y) = 0.49/0.14 = 3.418

------------[1]--------------
tensor(0, dtype=torch.int32)
tensor(0.7589)
tensor(0.9773)
P(c=0|y=0)0.7588800191879272
P(c=0|y=1)0.9772800207138062
P(c=1|y=0)0.24111999571323395
P(c=1|y=1)0.02271999977529049
P(...,c=0,..)/P(...,c=1,...) = 5.645610809326172
The MAP world is more likely then TAR
P(c=0)=1.4522783756256104, P(c=1)=0.25724026560783386)
P(c=0|x)=0.8721879720687866
P(c=0|y=1)=0.9772800207138062
P(c=1|y=1)=0.02271999977529049
map:0.98/1.4522783

In [43]:
concept_id = 0
num_samples = g.shape[0]

def p_c(concepts):
    p = torch.sum(g, dim=0)/num_samples
    return p
def p_y(labels):
    p = torch.sum(labels)/len(labels)
    return p.item()

pc = p_c(g)[concept_id]
py = p_y(labels)
print(pc,py)


tensor(0.1099) 0.415039986371994


# non functional

In [108]:
y = 0
    
def c_given_y(y):
    concepts_given_0 = []
    concepts_given_1 = []
    for i in range(num_samples):
        if labels[i] == 0:
            concepts_given_0.append(g[i])
        else:
            concepts_given_1.append(g[i])
    concepts_given_0 = torch.stack(concepts_given_0, dim=0)
    concepts_given_1 = torch.stack(concepts_given_1, dim=0)
    if y==0:
        return torch.sum(concepts_given_0,dim=0)/num_samples
    else:
        return torch.sum(concepts_given_1,dim=0)/num_samples

def y_given_c(y, concept_id, c):
    p_y1 = p_y(labels)
    p_y0 = 1-p_y1
    p_c1 = p_c(g)[concept_id].item()
    p_c0 = 1-p_c1
    if y == 0:
        p_y_given_c = c_given_y(y)[concept_id] * p_y0
    else:
        p_y_given_c = c_given_y(y)[concept_id] * p_y1
        
    if c == 0:
        p_y_given_c /= p_c0
    else:
        p_y_given_c /= p_c1
    print(f"Concept c={c} prob given y={y}", p_c(g)[concept_id].item())
    #print(c_given_y(y).shape)
    #print(p_y(labels))
    #print(p_c(g)[concept_id].item())
    return p_y_given_c

p_y_given_c = y_given_c(1,0,1)
print(p_y_given_c)

def c_given_x_y(y, concept_id, p_c1_given_x):
    normalization = y_given_c(y,concept_id, 0) * (1-p_c1_given_x) + y_given_c(y,concept_id,1) * p_c1_given_x
    return (y_given_c(y, concept_id, 1)*p_c1_given_x/normalization)



Concept c=1 prob given y=1 0.10992000252008438
tensor(0.4149)


In [110]:
concept_id = 0

refined_when_y_true = c_given_x_y(1,concept_id,dataset_train[0][1][concept_id])
refined_when_y_false = c_given_x_y(0,concept_id,dataset_train[0][1][concept_id])
print(f"When female {refined_when_y_false:.2f} | When male {refined_when_y_true:.2f}")
print("GP prediction for concept",dataset_train[0][1][concept_id].item())

Concept c=0 prob given y=1 0.10992000252008438
Concept c=1 prob given y=1 0.10992000252008438
Concept c=1 prob given y=1 0.10992000252008438
Concept c=0 prob given y=0 0.10992000252008438
Concept c=1 prob given y=0 0.10992000252008438
Concept c=1 prob given y=0 0.10992000252008438
When female 0.48 | When male 0.48
GP prediction for concept 0.10176625847816467
